## Silent bystander edits for predicting pegRNA efficiency with PRIDICT2.0

Silent bystander edits can increase the efficiency of (e)pegRNA efficiencies. This notebook allows to create inputs for PRIDICT2.0 that contain silent bystander edits, up to 5 bp up- and downstream of the edit.

Requirements:
- PRIDICT input sequence with **150 bp** context on both sides of the edit (e.g. 150bp - (A/G) - 150bp; longer context needed than the 100bp for standard PRIDICT2.0)
- PRIDICT2.0 conda environment (includes necessary packages)
- Replacements up to 5 bp allowed (5+ bp replacements will lead to too many options with silent bystanders to reasonably get PRIDICT2.0 predictions for all)

How to use:
- Use this notebook to create an input batch file for PRIDICT2.0 prediction with silent bystanders
- We only provide silent bystander predictions with **1bp replacements** or **multibp replacements** (not with insertion/deletions)
- If sequence is in exon, PRIDICT input sequence has to be *in frame* (ORF_start=0) and variable "silent" should be "yes"
- Keep variable "change_edit_bases" as default "no" if you do not wish to change your defined edit bases even if this would lead to the same amino acid.
  Example: NNN(GTA/CAT)NNN would be a V to H change, but same would be the case with NNN(GTA/CA**G**)NNN. If choosing "no" then changes within brackets will be kept and NNN(GTA/CA**G**)NNN would not be used.
- If you want to create any bystander (also non-silent) set variable "silent" to "no".

- Input: PRIDICT input format, but with 150bp flanking bases on both sides
- Single function: Create silent bystander PRIDICT inputs for 1 mutation/edit
- Batch function: Get the inputs for silent bystander for all PRIDICT inputs in an input .csv file
- Finally run PRIDICT2.0 (batch mode) with created input sequences to get efficiency predictions

Optional:
- Summarize predictions of all bystander-variants into a single file, by selecting best predicted pegRNA for each variant (last section)

### Import necessary packages

In [ ]:
import os
import pandas as pd

### Functions required to run notebook

In [ ]:
# All functions live in silent_bystander_input.py next to this notebook,
# so that they can also be imported as a module (outside of Jupyter):
#     from silent_bystander_input import silent_bystander_sequences

from silent_bystander_input import (
    generate_all_sequences,
    generate_combinations,
    convert_differences_to_lowercase,
    split_sequence,
    validate_context_length,
    process_contexts,
    handle_duplicate_sequences,
    isDNA,
    primesequenceparsing,
    bystander_creation_for_pridict,
    silent_bystander_sequences,
    bystander_input_generator,
)

### Single mode:

In [3]:
### Required variables (adapt to your needs)
name = 'test1_cftr'
pridict_input_original = 'TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTGGGGAAAAAAGGAAGAATTCTATTCTCAATCCAATCAACTCTATACGAAAATTTTCCATTGTGCAAAAGACTCCCTTACAAATGAATGGCATCGAAGAGGATTCT(G/C)ATGAGCCTTTAGAGAGAAGGCTGTCCTTAGTACCAGATTCTGAGCAGGGAGAGGCGATACTGCCTCGCATCAGCGTGATCAGCACTGGCCCCACGCTTCAGGCACGAAGGAGGCAGTCTGTCCTGAACCTGATGACACACTCAGTTAACC'
silent = 'yes'  # default: yes; put to 'no' if you want to get all possible bystander mutations, including non-silent mutations which change the AA sequence
change_edit_bases='no'  # default: no; put to 'yes' if you want to get all possible silent bystander mutations, including those which also change the edit bases you defined in the input (if applicable)
###

### default variables (do not change unless you want to adapt the script)
silent_surrounding_AA_nr = 2 # number of amino acids up- and downstream of edit AA for which silent bystander mutations will be created; default = 2; maximum 4 AA (up- and downstream) are allowed in this script
ORF_start = 0  # 0 means that the ORF starts at the first base of the input sequence; if the ORF starts at the second or third base, change this to 1 or 2, respectively
total_edit_limit = 40 # limit maximum length of edit (including bystander edits) to 40 bases; max. edit length for PRIDICT2.0 predictions is 40 bases
max_edit_length = 10 # maximum length of the edit bases you want to change
minimum_flanking = 94  # minimum edit-flanking length, after correcting for ORF_start. Only required as sanity check; do not change.
###

# run single bystander creation:
silentbystanderdf = bystander_creation_for_pridict(pridict_input_original, silent_surrounding_AA_nr, ORF_start, name, minimum_flanking, total_edit_limit,max_edit_length, silent=silent, change_edit_bases=change_edit_bases)


Number of silent options INCLUDING changed edit bases 191
Number of silent options EXCLUDING changed edit bases 191
Number of possible bystander mutations: 191
Sequence flanking edit position (before edit): GATTCTGATGAGCCT
Sequence flanking edit position (AFTER edit, without bystander editing): GATTCTcATGAGCCT
AA flanking edit position (before edit): DSDEP
AA flanking edit position (after edit, without bystander editing): DSHEP
AA flanking edit position (after edit, WITH bystander editing: {'DSHEP'}


In [4]:
# preview of silentbystanderdf:
silentbystanderdf.head(10)

,sequence_name,editseq,original_edit_length,final_edit_length_with_bystander,total_nr_of_base_changes,bystander_focus_sequence,bystander_focus_AA,editedonly_focus_sequence,editedonly_focus_AA
0,test1_cftr_GATGAG_cATGAa,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,6,2,GATTCTcATGAaCCT,DSHEP,GATTCTcATGAGCCT,DSHEP
1,test1_cftr_GATGAGCCT_cATGAaCCc,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,3,GATTCTcATGAaCCc,DSHEP,GATTCTcATGAGCCT,DSHEP
2,test1_cftr_GATGAGCCT_cATGAaCCa,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,3,GATTCTcATGAaCCa,DSHEP,GATTCTcATGAGCCT,DSHEP
3,test1_cftr_GATGAGCCT_cATGAaCCg,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,3,GATTCTcATGAaCCg,DSHEP,GATTCTcATGAGCCT,DSHEP
4,test1_cftr_GATGAGCCT_cATGAGCCc,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,2,GATTCTcATGAGCCc,DSHEP,GATTCTcATGAGCCT,DSHEP
5,test1_cftr_GATGAGCCT_cATGAGCCa,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,2,GATTCTcATGAGCCa,DSHEP,GATTCTcATGAGCCT,DSHEP
6,test1_cftr_GATGAGCCT_cATGAGCCg,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,2,GATTCTcATGAGCCg,DSHEP,GATTCTcATGAGCCT,DSHEP
7,test1_cftr_GATGAG_cAcGAa,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,6,3,GATTCTcAcGAaCCT,DSHEP,GATTCTcATGAGCCT,DSHEP
8,test1_cftr_GATGAGCCT_cAcGAaCCc,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,4,GATTCTcAcGAaCCc,DSHEP,GATTCTcATGAGCCT,DSHEP
9,test1_cftr_GATGAGCCT_cAcGAaCCa,TGGACAGAAACAAAAAAACAATCTTTTAAACAGACTGGAGAGTTTG...,1,9,4,GATTCTcAcGAaCCa,DSHEP,GATTCTcATGAGCCT,DSHEP


### Batch mode:

In [ ]:
# define input and output paths and filenames
inputpath = './input/'
inputfilename = 'input_testfile.csv' # check input_testfile.csv for details about formatting the input file; required columns: [Name, pridict_input, silent, change_edit_bases, in_frame]
outputpath = './output/'
outputfilename = 'outputfile.csv'
#

### --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
# default variables (do not change unless you want to adapt the script):
silent_surrounding_AA_nr = 2 # number of amino acids up- and downstream of edit AA for which silent bystander mutations will be created; default = 2; maximum 4 AA (up- and downstream) are allowed in this script
total_edit_limit = 40 # limit maximum length of edit (including bystander edits) to 40 bases; max. edit length for PRIDICT2.0 predictions is 40 bases
max_edit_length = 10 # maximum length of the edit bases you want to change
minimum_flanking = 94  # minimum edit-flanking length, after correcting for ORF_start. Only required as sanity check; do not change.
### --------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# run bystander input generator
# (to get the same sequences as an in-memory iterable instead of a csv file, use
#  silent_bystander_sequences(pridict_input, name=..., silent=..., change_edit_bases=...) )
inputfiledf, outputfiledf = bystander_input_generator(inputpath, inputfilename, outputpath, outputfilename, silent_surrounding_AA_nr, total_edit_limit, max_edit_length, minimum_flanking)

# continue with running PRIDICT2 (outside of this notebook) with the outputfile.csv as input file
# example command: python pridict2_pegRNA_design.py batch --input-dir ./addons/silentbystander/output --input-fname outputfile.csv --output-dir ./predictions

# Optional but NOT RECOMMENDED: run PRIDICT2 from within this notebook. (uncomment !python command below)
# Caveat: takes a LONG time to run; we recommend running it separately via commandline
# !python ../../pridict2_pegRNA_design.py batch --input-dir ./output --input-fname outputfile.csv --output-dir ../../predictions

### Summarize PRIDICT2.0 predictions of silent bystanders after running PRIDICT2.0
- Only run this after you ran PRIDICT2.0 with the output batch file created above. 

- The code below summarizes all the predictions with different silent bystanders in one file, sorts this by K562 score and saves it as summary prediction file.

- For MMR-deficient context, change "sort_value" from "K562" to "HEK".

- From this summary file, we suggest to take e.g. the top 5 pegRNAs and test these in your experimental setup

In [ ]:
### Summarize PRIDICT2 predictions:
pridict2_predictions_folder = '../predictions/'  # folder with all PRIDICT2 predictions (default "../predictions/")
summary_prediction_output_folder = './summarized_silent_pridict2_predictions/' # folder where summarized prediction files will be saved
sort_value = 'K562' # change to "HEK" for sorting to MMR-deficient cell line prediction

# filelist of all .csv files in pridict2_predictions_folder:
filelist = [f for f in os.listdir(pridict2_predictions_folder) if f.endswith('pegRNA_Pridict_full.csv')]

for index, row in inputfiledf.iterrows():
    sequence_name = row['Name']
    # get all files in filelist that start with the sequence_name
    sequence_files = [f for f in filelist if f.startswith(sequence_name)]
    # read all files and concatenate them
    all_files = []
    for file in sequence_files:
        all_files.append(pd.read_csv(pridict2_predictions_folder+file))
    all_files_df = pd.concat(all_files, ignore_index=True)
    # sort all_files_df by column "PRIDICT2_0_editing_Score_deep_..." (from highest to lowest)
    all_files_df = all_files_df.sort_values(by='PRIDICT2_0_editing_Score_deep_'+sort_value, ascending=False)
    # save concatenated file
    all_files_df.to_csv(summary_prediction_output_folder+sequence_name+'_all_silent_predictions.csv')